In [ ]:

import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

# 1. Load dataset
data = pd.read_csv('AutoEncoder/ECG/ecg.csv', header=None)

X = data.iloc[:, :-1]
y = data.iloc[:, -1]

# 2. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Preprocess data
X_train = X_train[y_train == 0]

minimum = X_train.min().min()
maximum = X_train.max().max()

X_train = (X_train - minimum) / (maximum - minimum + 1e-8)
X_test = (X_test - minimum) / (maximum - minimum + 1e-8)

# 4. Build Autoencoder
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(140,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(140, activation='sigmoid')
])

# 5. Compile model
model.compile(optimizer='adam', loss='mse')

# 6. Train model
history = model.fit(
    X_train, X_train,
    epochs=10,
    batch_size=128
)

# 7. Calculate error and detect anomalies
train_pred = model.predict(X_train)
train_error = np.mean((X_train - train_pred)**2, axis=1)

threshold = np.percentile(train_error, 95)

test_pred = model.predict(X_test)
test_error = np.mean((X_test - test_pred)**2, axis=1)

y_pred = (test_error > threshold).astype(int)
y_true = (y_test != 0).astype(int)

# 8. Evaluate and plot
print("Threshold:", threshold)

plt.plot(history.history['loss'])
plt.title('Autoencoder Loss')
plt.show()

ConfusionMatrixDisplay.from_predictions(y_true, y_pred)
plt.show()
